In [ ]:
import numpy as np
words = open('names.txt').read().splitlines()   # from github.com/karpathy/makemore
assert not any(' ' in w for w in words)

chars = sorted(set(''.join(words)))
stoi = {c: i+1 for i, c in enumerate(chars)}
stoi[' '] = 0
itos = {i: c for c, i in stoi.items()}
vocab_size = len(stoi)                          # 27

context_len = 3
X_list, y_list = [], []
for w in words:
    context = [0] * context_len
    for ch in w + ' ':
        X_list.append(context.copy())
        y_list.append(stoi[ch])
        context = context[1:] + [stoi[ch]]
X = np.array(X_list) / 26.0
X_oh = np.eye(vocab_size)[np.array(X_list)].reshape(len(X_list), -1)   # (228146, 81)
Y = np.eye(vocab_size)[np.array(y_list)]                               # (228146, 27)

FileNotFoundError: [Errno 2] No such file or directory: 'names.txt'

In [ ]:
class Affine:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def forward(self, x):
        return x @ self.weight.T + self.bias



In [ ]:
class Sigmoid:
    def __init__(self):
        pass

    def forward(self, x):
        return 1 / (np.exp(-x) + 1)


In [ ]:
class Softmax:
  def __init__(self):
    pass

  def forward(self, x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)



In [ ]:
class MLP_name:
    def __init__(self, arc, out):
        self.g = np.empty(len(arc), dtype=object)
        for i in range(len(arc)):
            if(i != len(arc) - 1):
                self.g[i] = [np.random.randn(arc[i + 1], arc[i]) * np.sqrt(1/arc[i]), np.zeros((1, arc[i + 1]))]
            else:
                self.g[i] = [np.random.randn(out, arc[i]) * np.sqrt(1/arc[i]), np.zeros((1, out))]
        self.update()
    def forward(self, x):
        for layer in self.layers:
            for module in layer:
                x =  module.forward(x)
        return x.astype(float)
    def update(self):
        self.n = len(self.g)
        self.affine = np.empty(self.n, dtype=object)
        self.activation = np.empty(self.n, dtype=object)
        self.layer = np.empty(self.n, dtype=list)
        self.layers = [None] * self.n

        for i in range(self.n):
            self.affine[i] = Affine(self.g[i][0], self.g[i][1])
            if(i != self.n - 1):
                self.activation[i] = Sigmoid()
            else:
                self.activation[i] = Softmax()
            self.layer[i] = [self.affine[i], self.activation[i]]
            self.layers[i] = (self.layer[i])



    def gd(self, X_train, y_train, lr):
        self.dgd(X_train, y_train, lr)


    def train(self, X_train, y_train, epochs, lr):
        for i in range(epochs):
            self.gd(X_train, y_train, lr)


    def test(self, X_test, y_test):
        cr = 0
        for i in range(len(X_test)):
            cr +=  (np.argmax(self.forward(X_test[i])) == y_test[i])

        return cr/len(X_test)



    def dgd(self, x, real, lr):
        xc = x
        n = len(x)
        ins = []
        outs = [0]
        for layer in self.layers:
            ins.append(x)
            for module in layer:
                x =  module.forward(x)
            outs.append(x)
        diff = x - real

        rolling = diff
        for i in range(self.n - 1, -1, -1):
            self.g[i][1] -= lr / n * rolling.sum(axis=0)
            ww = self.g[i][0].copy()
            self.g[i][0] -= lr / n * (rolling.T @  ins[i])
            rolling = rolling @ ww
            if i > 0:
                rolling *= (outs[i]) * (1 - outs[i])
        return None



    def __call__(self, x):
        return self.forward(x)

In [ ]:
model2 = MLP_name([81, 128, 128], 27)

model2.train(X_oh, Y, epochs=100, lr=0.1)





In [ ]:
def one_hot(idx, size=27):
    v = np.zeros(size)
    v[idx] = 1.0
    return v

def un_hot(v):
    return int(np.argmax(v))

In [ ]:
st = ''
s = [0, 0, 0]
while True:
    x = model2.forward(np.eye(27)[s].reshape(1, -1))
    #print(x)
    q = np.zeros(len(x[0]))
    for i in range(len(x[0])):
        q[i] = x[0][i]

    l = np.random.choice(len(q), p=q)
    if l == 26:
        continue
    if l == 0:
        break
    s = np.append(s, l)
    s = s[1:]
    st = st + itos[l]
print(st)


In [ ]:
model = MLP_name([3, 128, 128], 27)

model.train(X, Y, epochs=100, lr=0.1)





In [ ]:
st = ''
s = [0, 0, 0]
while True:
    x = model.forward(s)
    #print(x)
    q = np.zeros(len(x[0]))
    for i in range(len(x[0])):
        q[i] = x[0][i]

    l = np.random.choice(len(q), p=q)
    if l == 26:
        continue
    if l == 0:
        break
    s = np.append(s, l)
    s = s[1:]
    st = st + itos[l]
print(st)
